In [1]:
print("hello")

hello


In [2]:
%pip install numpy pandas

  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
    --------------------------------------- 0.3/12.4 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.4 MB 3.1 MB/s eta 0:00:04
   ----- ---------------------------------- 1.6/12.4 MB 3.4 MB/s eta 0:00:04
   ------- -------------------------------- 2.4/12.4 MB 3.4 MB/s eta 0:00:04
   ----------- ---------------------------- 3.7/12.4 MB 3.8 MB/s eta 0:00:03
   --------------- ------------------------ 4.7/12.4 MB 4.2 MB/s eta 0:00:02
   ------------------- -------------------- 6.0/12.4 MB 4.4 MB/s eta 0:00:02
   ----------------------- ---------------- 7.3/12.4 MB 4.7 MB/s eta 0:00:02
   ----------------------------- ---------- 9.2/12.4 MB 5.2 MB/s eta 0:00:01
   ----------------------------------- ---- 11.0/12.4 MB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 12.4/12.4 MB 5.8 MB/s eta 0:00:00
   -----------------


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ============================================================
# DVLOG SETUP
# DVlog has acoustic.npy and visual.npy per sample
# Labels from labels.csv: index, label, duration, gender, fold
# Path: C:\Users\giria\Downloads\dvlog-dataset
# On your Linux machine this is likely under ~/Downloads/
# ============================================================

import os
import numpy as np
import pandas as pd

DVLOG_ROOT = r"C:\Users\giria\Downloads\dvlog-dataset"
DVLOG_LABELS = os.path.join(DVLOG_ROOT, "labels.csv")

dvlog_df = pd.read_csv(DVLOG_LABELS)
print(f"DVlog samples: {len(dvlog_df)}")
print(f"Columns: {dvlog_df.columns.tolist()}")
print(f"\nLabel dist:\n{dvlog_df['label'].value_counts()}")
print(f"\nFold dist:\n{dvlog_df['fold'].value_counts()}")

# Convert to binary: depression=1, normal=0
# Convert gender: f=1, m=0  (matches DAIC-WOZ convention)
dvlog_df['label_bin']  = (dvlog_df['label'] == 'depression').astype(int)
dvlog_df['gender_bin'] = (dvlog_df['gender'] == 'f').astype(int)

print(f"\nSample row:")
print(dvlog_df[['index','label','label_bin',
                'gender','gender_bin','fold']].head(5))

DVlog samples: 961
Columns: ['index', 'label', 'duration', 'gender', 'fold']

Label dist:
label
depression    555
normal        406
Name: count, dtype: int64

Fold dist:
fold
train    647
test     212
valid    102
Name: count, dtype: int64

Sample row:
   index       label  label_bin gender  gender_bin   fold
0      0  depression          1      f           1  train
1      1  depression          1      f           1   test
2      2  depression          1      m           0  train
3      3  depression          1      m           0  valid
4      4  depression          1      f           1  train


In [4]:
import os

print("DVLOG_ROOT =", DVLOG_ROOT)
print()

items = os.listdir(DVLOG_ROOT)

print("Total items:", len(items))
print()
print("First 30 items:")
print(items[:30])

DVLOG_ROOT = C:\Users\giria\Downloads\dvlog-dataset

Total items: 962

First 30 items:
['0', '1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '12', '120', '121', '122', '123', '124']


In [5]:
import os

print(os.listdir(os.path.join(DVLOG_ROOT, "0")))

['0_acoustic.npy', '0_visual.npy']


In [6]:
sample_idx = 0

folder = os.path.join(DVLOG_ROOT, str(sample_idx))

acoustic_path = os.path.join(folder, f"{sample_idx}_acoustic.npy")
visual_path   = os.path.join(folder, f"{sample_idx}_visual.npy")

acoustic = np.load(acoustic_path)
visual   = np.load(visual_path)

print(f"Sample {sample_idx}")
print("Acoustic:", acoustic.shape, acoustic.dtype)
print("Visual   :", visual.shape, visual.dtype)

print("Acoustic range:", acoustic.min(), acoustic.max())
print("Visual range   :", visual.min(), visual.max())

Sample 0
Acoustic: (823, 25) float64
Visual   : (823, 136) float64
Acoustic range: -201.0 2876.752685546875
Visual range   : -3.1552703733276792 2.8355472203777


In [7]:
import os

idx = 0
folder = os.path.join(DVLOG_ROOT, str(idx))

print("Folder exists :", os.path.isdir(folder))
print("Folder path   :", folder)

print("\nContents:")
print(os.listdir(folder))

a_path = os.path.join(folder, f"{idx}_acoustic.npy")
v_path = os.path.join(folder, f"{idx}_visual.npy")

print("\nAcoustic path:", a_path)
print("Exists?", os.path.exists(a_path))

print("\nVisual path:", v_path)
print("Exists?", os.path.exists(v_path))

Folder exists : True
Folder path   : C:\Users\giria\Downloads\dvlog-dataset\0

Contents:
['0_acoustic.npy', '0_visual.npy']

Acoustic path: C:\Users\giria\Downloads\dvlog-dataset\0\0_acoustic.npy
Exists? True

Visual path: C:\Users\giria\Downloads\dvlog-dataset\0\0_visual.npy
Exists? True


In [8]:
# ============================================================
# DVLOG FEATURE LOADING
#
# DVlog has:
#   acoustic.npy: eGeMAPS features, sampled at 1Hz (per second)
#   visual.npy  : face landmarks from Dlib, sampled at 1Hz
#
# DAIC-WOZ has:
#   audio   : 100Hz → 600 frames per 6s window
#   video   :  25Hz → 150 frames per 6s window
#
# DVlog is 1Hz so 6s window = 6 frames only — too short.
# We use 30-second windows instead:
#   acoustic: 1Hz × 30s = 30 frames
#   visual  : 1Hz × 30s = 30 frames
#
# Strategy: map DVlog features onto GATA-Dep's 5 modalities:
#   audio    ← acoustic features (eGeMAPS, padded/projected to 79D)
#   au       ← zeros (not available in DVlog)
#   landmark ← visual features (Dlib landmarks, padded to 204D)
#   gaze     ← zeros (not available in DVlog)
#   pose     ← zeros (not available in DVlog)
#
# The GATA model is retrained from scratch on DVlog
# with new encoder dims matching DVlog features.
# ============================================================

DVLOG_WINDOW_SEC = 30   # seconds per window
DVLOG_HZ         = 1    # DVlog is sampled at 1Hz
DVLOG_WIN_FRAMES = DVLOG_WINDOW_SEC * DVLOG_HZ  # 30 frames


def normalize_dvlog(arr):
    """Zero-mean unit-variance per column."""
    mean = np.nanmean(arr, axis=0, keepdims=True)
    std  = np.nanstd (arr, axis=0, keepdims=True)
    std[std == 0] = 1.0
    arr = (arr - mean) / std
    return np.nan_to_num(
        arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def load_dvlog_sample(idx, dvlog_root):
    """Load acoustic and visual features for one DVlog sample."""

    folder = os.path.join(dvlog_root, str(idx))

    a_path = os.path.join(folder, f"{idx}_acoustic.npy")
    v_path = os.path.join(folder, f"{idx}_visual.npy")

    if not os.path.exists(a_path):
        return None

    if not os.path.exists(v_path):
        return None

    acoustic = np.load(a_path).astype(np.float32)
    visual   = np.load(v_path).astype(np.float32)

    # Ensure 2D: [T,D]
    if acoustic.ndim == 1:
        acoustic = acoustic[:, None]

    if visual.ndim == 1:
        visual = visual[:, None]

    return {
        "acoustic": acoustic,
        "visual": visual
    }


def make_dvlog_windows(features, win_frames=DVLOG_WIN_FRAMES):
    """Split DVlog features into non-overlapping windows."""
    acoustic = normalize_dvlog(features['acoustic'])
    visual   = normalize_dvlog(features['visual'])

    # Align lengths
    min_len  = min(len(acoustic), len(visual))
    acoustic = acoustic[:min_len]
    visual   = visual[:min_len]

    n_wins = min_len // win_frames
    windows = []
    for i in range(n_wins):
        s = i * win_frames
        windows.append({
            'acoustic': acoustic[s: s + win_frames],
            'visual':   visual  [s: s + win_frames],
        })
    return windows


# Test on sample 0
# Test on sample 0
_feat = load_dvlog_sample(0, DVLOG_ROOT)

if _feat is None:
    raise RuntimeError("Failed to load DVlog sample 0")

print(f"Sample 0: acoustic={_feat['acoustic'].shape}")
print(f"Sample 0: visual={_feat['visual'].shape}")

_wins = make_dvlog_windows(_feat)
print(f"Windows: {len(_wins)}")

if len(_wins):
    print(f"Window acoustic: {_wins[0]['acoustic'].shape}")
    print(f"Window visual:   {_wins[0]['visual'].shape}")

DVLOG_ACOUSTIC_DIM = _feat["acoustic"].shape[1]
DVLOG_VISUAL_DIM   = _feat["visual"].shape[1]

print(f"\nDVlog acoustic dim: {DVLOG_ACOUSTIC_DIM}")
print(f"DVlog visual dim: {DVLOG_VISUAL_DIM}")

Sample 0: acoustic=(823, 25)
Sample 0: visual=(823, 136)
Windows: 27
Window acoustic: (30, 25)
Window visual:   (30, 136)

DVlog acoustic dim: 25
DVlog visual dim: 136


In [9]:
# ============================================================
# CACHE DVLOG — save all windows as .npy files
# ============================================================

DVLOG_CACHE_DIR = os.path.expanduser("~/dvlog_cache")
os.makedirs(DVLOG_CACHE_DIR, exist_ok=True)

def cache_dvlog(dvlog_df, dvlog_root, cache_dir):
    already = set(os.listdir(cache_dir))
    total_windows = 0
    failed = []

    for _, row in dvlog_df.iterrows():
        idx    = int(row['index'])
        fname  = f"{idx}.npy"
        if fname in already:
            continue

        feats = load_dvlog_sample(idx, dvlog_root)
        if feats is None:
            failed.append(idx); continue

        wins = make_dvlog_windows(feats)
        if not wins:
            failed.append(idx); continue

        # Shape check
        if wins[0]['acoustic'].shape[1] == 0 or \
           wins[0]['visual'].shape[1] == 0:
            failed.append(idx); continue

        np.save(
            os.path.join(cache_dir, fname),
            {
                'acoustic': np.stack([w['acoustic'] for w in wins]),
                'visual':   np.stack([w['visual']   for w in wins]),
                'label':    np.array([int(row['label_bin'])]  * len(wins)),
                'gender':   np.array([int(row['gender_bin'])] * len(wins)),
            }
        )
        total_windows += len(wins)

    size_mb = sum(
        os.path.getsize(os.path.join(cache_dir, f))
        for f in os.listdir(cache_dir)
    ) / 1e6
    print(f"Cached {total_windows} windows from "
          f"{len(dvlog_df)-len(failed)} samples")
    print(f"Failed: {len(failed)}  Cache size: {size_mb:.1f} MB")

print("Caching DVlog...")
cache_dvlog(dvlog_df, DVLOG_ROOT, DVLOG_CACHE_DIR)

Caching DVlog...
Cached 0 windows from 959 samples
Failed: 2  Cache size: 360.4 MB


In [21]:
%pip uninstall -y torch torchvision torchaudio

Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 GB 3.4 MB/s eta 0:12:34
   ---------------------------------------- 0.0/2.5 GB 8.4 MB/s eta 0:05:02
   ---------------------------------------- 0.0/2.5 GB 10.3 MB/s eta 0:04:06
   ---------------------------------------- 0.0/2.5 GB 10.4 MB/s eta 0:04:02
   ---------------------------------------- 0.0/2.5 GB 11.3 MB/s eta 0:03:44
   ---------------------------------------- 0.0/2.5 GB 11.6 MB/s eta 0:03:38
   ---------------------------------------- 0.0/2.5 GB 11.4 MB/s eta 0:03:40
   ---------------------------------------- 0.0/2


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import torch

if torch.cuda.is_available():
    print(f"Success! Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU still not recognized. Check your installation.")

Success! Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [13]:
# ============================================================
# DVLOG RAM DATASET
# Loads all windows into RAM — same pattern as RAMDAICDataset
# Returns same 7-tuple format so training loop works unchanged
# ============================================================

class DVlogRAMDataset(Dataset):
    def __init__(self, cache_dir, indices):
        acoustic_list = []; visual_list = []
        labels = []; genders = []; pids_all = []

        for idx in indices:
            path = os.path.join(cache_dir, f"{idx}.npy")
            if not os.path.exists(path):
                continue
            try:
                data = np.load(path, allow_pickle=True).item()
                n = len(data['label'])
                acoustic_list.append(data['acoustic'])
                visual_list.append(data['visual'])
                labels.extend(data['label'].tolist())
                genders.extend(data['gender'].tolist())
                pids_all.extend([idx] * n)
            except Exception as e:
                print(f"  SKIP {idx}: {e}")

        self.acoustic = torch.FloatTensor(np.concatenate(acoustic_list))
        self.visual   = torch.FloatTensor(np.concatenate(visual_list))
        self.labels   = torch.tensor(labels,   dtype=torch.long)
        self.genders  = torch.tensor(genders,  dtype=torch.long)
        self.pids     = torch.tensor(pids_all, dtype=torch.long)

        dep  = self.labels.sum().item()
        ctrl = len(self.labels) - dep
        ram  = (self.acoustic.nbytes + self.visual.nbytes) / 1e9
        print(f"  {len(indices)} samples → {len(self.labels)} windows "
              f"dep={int(dep)} ctrl={ctrl} RAM={ram:.2f}GB")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Returns: acoustic, visual, label, gender, pid
        return (self.acoustic[idx], self.visual[idx],
                self.labels[idx], self.genders[idx],
                self.pids[idx])

NameError: name 'Dataset' is not defined

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [15]:
import torch
import torch.nn as nn

class ModalityEncoder(nn.Module):
    """
    Projects any input feature dimension to the common GATA embedding size.
    Input : [B, T, input_dim]
    Output: [B, T, 256]
    """
    def __init__(self, input_dim, d_model=256, dropout=0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [16]:
# ============================================================
# IMPORTS + DEVICE SETUP
# Run this ONCE at the top of the notebook
# ============================================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
)

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("PyTorch :", torch.__version__)
print("Device  :", device)

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print(
        "VRAM    : %.2f GB"
        % (torch.cuda.get_device_properties(0).total_memory / 1024**3)
    )

print("=" * 60)

ModuleNotFoundError: No module named 'sklearn'

In [17]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

2.6.0+cu124
True
12.4
NVIDIA GeForce RTX 3050 Laptop GPU


In [18]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())

2.6.0+cu124
True
12.4
1


In [19]:
# ============================================================
# GATA MODEL FOR DVLOG
#
# DVlog only has 2 modalities: acoustic + visual
# We adapt GATAFusion to use only 2 attention modules
# and 2 offset weight rows instead of 4
# ============================================================

class GATAFusionDVlog(nn.Module):
    """GATA fusion for DVlog: acoustic (query) + visual (key/val)."""
    def __init__(self, d_model=256, n_heads=4, K=4, dropout=0.1):
        super().__init__()
        self.K         = K
        self.n_offsets = 2 * K + 1

        # Only one cross-modal attention: acoustic queries visual
        self.attn_visual = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)

        # offset_weights: [2=gender, 1=modality, n_offsets]
        self.offset_weights = nn.Parameter(
            torch.zeros(2, 1, self.n_offsets))

        self.norm_visual = nn.LayerNorm(d_model)
        self.output_proj = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(), nn.Dropout(dropout))

    def _shift(self, x, k):
        if k == 0: return x
        B, T, D = x.shape
        out = torch.zeros_like(x)
        if k > 0:
            out[:, k:, :]    = x[:, :T-k, :]
        else:
            ak = abs(k)
            out[:, :T-ak, :] = x[:, ak:, :]
        return out

    def forward(self, acoustic, visual, gender):
        B, T, D = acoustic.shape
        n = self.n_offsets

        # Shifted visual representations
        shifted = torch.stack(
            [self._shift(visual, k)
             for k in range(-self.K, self.K+1)],
            dim=1).reshape(B * n, T, D)

        q_rep = (acoustic.unsqueeze(1)
                         .expand(-1, n, -1, -1)
                         .reshape(B * n, T, D))

        attn_out, _ = self.attn_visual(
            query=q_rep, key=shifted, value=shifted)
        attn_out = attn_out.view(B, n, T, D)

        w_all    = F.softmax(
            self.offset_weights[:, 0, :], dim=-1)
        w_sample = w_all[gender].unsqueeze(-1).unsqueeze(-1)

        visual_aligned = (attn_out * w_sample).sum(dim=1)
        visual_aligned = self.norm_visual(acoustic + visual_aligned)

        # Concatenate acoustic + aligned visual
        combined = torch.cat([acoustic, visual_aligned], dim=-1)
        return self.output_proj(combined)


class GATADepDVlog(nn.Module):
    """Full GATA-Dep model for DVlog with 2 modalities."""
    def __init__(self, acoustic_dim, visual_dim,
                 d_model=256, n_heads=4, n_layers=4,
                 K=4, dropout=0.1, num_classes=2):
        super().__init__()

        self.enc_acoustic = ModalityEncoder(acoustic_dim)
        self.enc_visual   = ModalityEncoder(visual_dim)

        self.gata = GATAFusionDVlog(d_model, n_heads, K, dropout)

        self.mc_dropout = nn.Dropout(p=0.3)

        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8,
            dim_feedforward=512, dropout=dropout,
            batch_first=True)
        self.transformer = nn.TransformerEncoder(
            enc, num_layers=n_layers)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, acoustic, visual, gender, mc_samples=1):
        a = self.enc_acoustic(acoustic)   # [B, T, 256]
        v = self.enc_visual(visual)       # [B, T, 256]

        a = self.mc_dropout(a)
        v = self.mc_dropout(v)

        fused  = self.gata(a, v, gender)
        out    = self.transformer(fused).mean(dim=1)
        return self.classifier(out)


# Build model
dvlog_gata = GATADepDVlog(
    acoustic_dim = DVLOG_ACOUSTIC_DIM,
    visual_dim   = DVLOG_VISUAL_DIM,
    K=4
)
dvlog_gata = dvlog_gata.to(device)

n_params = sum(p.numel() for p in dvlog_gata.parameters()
               if p.requires_grad)
print(f"DVlog GATA-Dep parameters: {n_params:,}")

NameError: name 'device' is not defined

In [21]:
# ============================================================
# CACHE DVLOG — save all windows as .npy files
# ============================================================

DVLOG_CACHE_DIR = os.path.expanduser("~/dvlog_cache")
os.makedirs(DVLOG_CACHE_DIR, exist_ok=True)

def cache_dvlog(dvlog_df, dvlog_root, cache_dir):
    already = set(os.listdir(cache_dir))
    total_windows = 0
    failed = []

    for _, row in dvlog_df.iterrows():
        idx    = int(row['index'])
        fname  = f"{idx}.npy"
        if fname in already:
            continue

        feats = load_dvlog_sample(idx, dvlog_root)
        if feats is None:
            failed.append(idx); continue

        wins = make_dvlog_windows(feats)
        if not wins:
            failed.append(idx); continue

        # Shape check
        if wins[0]['acoustic'].shape[1] == 0 or \
           wins[0]['visual'].shape[1] == 0:
            failed.append(idx); continue

        np.save(
            os.path.join(cache_dir, fname),
            {
                'acoustic': np.stack([w['acoustic'] for w in wins]),
                'visual':   np.stack([w['visual']   for w in wins]),
                'label':    np.array([int(row['label_bin'])]  * len(wins)),
                'gender':   np.array([int(row['gender_bin'])] * len(wins)),
            }
        )
        total_windows += len(wins)

    size_mb = sum(
        os.path.getsize(os.path.join(cache_dir, f))
        for f in os.listdir(cache_dir)
    ) / 1e6
    print(f"Cached {total_windows} windows from "
          f"{len(dvlog_df)-len(failed)} samples")
    print(f"Failed: {len(failed)}  Cache size: {size_mb:.1f} MB")

print("Caching DVlog...")
cache_dvlog(dvlog_df, DVLOG_ROOT, DVLOG_CACHE_DIR)

Caching DVlog...
Cached 0 windows from 959 samples
Failed: 2  Cache size: 360.4 MB


In [20]:
# ============================================================
# DVLOG RAM DATASET
# Loads all windows into RAM — same pattern as RAMDAICDataset
# Returns same 7-tuple format so training loop works unchanged
# ============================================================

class DVlogRAMDataset(Dataset):
    def __init__(self, cache_dir, indices):
        acoustic_list = []; visual_list = []
        labels = []; genders = []; pids_all = []

        for idx in indices:
            path = os.path.join(cache_dir, f"{idx}.npy")
            if not os.path.exists(path):
                continue
            try:
                data = np.load(path, allow_pickle=True).item()
                n = len(data['label'])
                acoustic_list.append(data['acoustic'])
                visual_list.append(data['visual'])
                labels.extend(data['label'].tolist())
                genders.extend(data['gender'].tolist())
                pids_all.extend([idx] * n)
            except Exception as e:
                print(f"  SKIP {idx}: {e}")

        self.acoustic = torch.FloatTensor(np.concatenate(acoustic_list))
        self.visual   = torch.FloatTensor(np.concatenate(visual_list))
        self.labels   = torch.tensor(labels,   dtype=torch.long)
        self.genders  = torch.tensor(genders,  dtype=torch.long)
        self.pids     = torch.tensor(pids_all, dtype=torch.long)

        dep  = self.labels.sum().item()
        ctrl = len(self.labels) - dep
        ram  = (self.acoustic.nbytes + self.visual.nbytes) / 1e9
        print(f"  {len(indices)} samples → {len(self.labels)} windows "
              f"dep={int(dep)} ctrl={ctrl} RAM={ram:.2f}GB")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Returns: acoustic, visual, label, gender, pid
        return (self.acoustic[idx], self.visual[idx],
                self.labels[idx], self.genders[idx],
                self.pids[idx])

SyntaxError: incomplete input (438425707.py, line 9)

In [24]:
# ============================================================
# GATA MODEL FOR DVLOG
#
# DVlog only has 2 modalities: acoustic + visual
# We adapt GATAFusion to use only 2 attention modules
# and 2 offset weight rows instead of 4
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class GATAFusionDVlog(nn.Module):
    """GATA fusion for DVlog: acoustic (query) + visual (key/val)."""
    def __init__(self, d_model=256, n_heads=4, K=4, dropout=0.1):
        super().__init__()
        self.K         = K
        self.n_offsets = 2 * K + 1

        # Only one cross-modal attention: acoustic queries visual
        self.attn_visual = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)

        # offset_weights: [2=gender, 1=modality, n_offsets]
        self.offset_weights = nn.Parameter(
            torch.zeros(2, 1, self.n_offsets))

        self.norm_visual = nn.LayerNorm(d_model)
        self.output_proj = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(), nn.Dropout(dropout))

    def _shift(self, x, k):
        if k == 0: return x
        B, T, D = x.shape
        out = torch.zeros_like(x)
        if k > 0:
            out[:, k:, :]    = x[:, :T-k, :]
        else:
            ak = abs(k)
            out[:, :T-ak, :] = x[:, ak:, :]
        return out

    def forward(self, acoustic, visual, gender):
        B, T, D = acoustic.shape
        n = self.n_offsets

        # Shifted visual representations
        shifted = torch.stack(
            [self._shift(visual, k)
             for k in range(-self.K, self.K+1)],
            dim=1).reshape(B * n, T, D)

        q_rep = (acoustic.unsqueeze(1)
                         .expand(-1, n, -1, -1)
                         .reshape(B * n, T, D))

        attn_out, _ = self.attn_visual(
            query=q_rep, key=shifted, value=shifted)
        attn_out = attn_out.view(B, n, T, D)

        w_all    = F.softmax(
            self.offset_weights[:, 0, :], dim=-1)
        w_sample = w_all[gender].unsqueeze(-1).unsqueeze(-1)

        visual_aligned = (attn_out * w_sample).sum(dim=1)
        visual_aligned = self.norm_visual(acoustic + visual_aligned)

        # Concatenate acoustic + aligned visual
        combined = torch.cat([acoustic, visual_aligned], dim=-1)
        return self.output_proj(combined)


class GATADepDVlog(nn.Module):
    """Full GATA-Dep model for DVlog with 2 modalities."""
    def __init__(self, acoustic_dim, visual_dim,
                 d_model=256, n_heads=4, n_layers=4,
                 K=4, dropout=0.1, num_classes=2):
        super().__init__()

        self.enc_acoustic = ModalityEncoder(acoustic_dim)
        self.enc_visual   = ModalityEncoder(visual_dim)

        self.gata = GATAFusionDVlog(d_model, n_heads, K, dropout)

        self.mc_dropout = nn.Dropout(p=0.3)

        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8,
            dim_feedforward=512, dropout=dropout,
            batch_first=True)
        self.transformer = nn.TransformerEncoder(
            enc, num_layers=n_layers)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, acoustic, visual, gender, mc_samples=1):
        a = self.enc_acoustic(acoustic)   # [B, T, 256]
        v = self.enc_visual(visual)       # [B, T, 256]

        a = self.mc_dropout(a)
        v = self.mc_dropout(v)

        fused  = self.gata(a, v, gender)
        out    = self.transformer(fused).mean(dim=1)
        return self.classifier(out)


# Build model
dvlog_gata = GATADepDVlog(
    acoustic_dim = DVLOG_ACOUSTIC_DIM,
    visual_dim   = DVLOG_VISUAL_DIM,
    K=4
)
dvlog_gata = dvlog_gata.to(device)

n_params = sum(p.numel() for p in dvlog_gata.parameters()
               if p.requires_grad)
print(f"DVlog GATA-Dep parameters: {n_params:,}")# ============================================================
# GATA MODEL FOR DVLOG
#
# DVlog only has 2 modalities: acoustic + visual
# We adapt GATAFusion to use only 2 attention modules
# and 2 offset weight rows instead of 4
# ============================================================

class GATAFusionDVlog(nn.Module):
    """GATA fusion for DVlog: acoustic (query) + visual (key/val)."""
    def __init__(self, d_model=256, n_heads=4, K=4, dropout=0.1):
        super().__init__()
        self.K         = K
        self.n_offsets = 2 * K + 1

        # Only one cross-modal attention: acoustic queries visual
        self.attn_visual = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)

        # offset_weights: [2=gender, 1=modality, n_offsets]
        self.offset_weights = nn.Parameter(
            torch.zeros(2, 1, self.n_offsets))

        self.norm_visual = nn.LayerNorm(d_model)
        self.output_proj = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(), nn.Dropout(dropout))

    def _shift(self, x, k):
        if k == 0: return x
        B, T, D = x.shape
        out = torch.zeros_like(x)
        if k > 0:
            out[:, k:, :]    = x[:, :T-k, :]
        else:
            ak = abs(k)
            out[:, :T-ak, :] = x[:, ak:, :]
        return out

    def forward(self, acoustic, visual, gender):
        B, T, D = acoustic.shape
        n = self.n_offsets

        # Shifted visual representations
        shifted = torch.stack(
            [self._shift(visual, k)
             for k in range(-self.K, self.K+1)],
            dim=1).reshape(B * n, T, D)

        q_rep = (acoustic.unsqueeze(1)
                         .expand(-1, n, -1, -1)
                         .reshape(B * n, T, D))

        attn_out, _ = self.attn_visual(
            query=q_rep, key=shifted, value=shifted)
        attn_out = attn_out.view(B, n, T, D)

        w_all    = F.softmax(
            self.offset_weights[:, 0, :], dim=-1)
        w_sample = w_all[gender].unsqueeze(-1).unsqueeze(-1)

        visual_aligned = (attn_out * w_sample).sum(dim=1)
        visual_aligned = self.norm_visual(acoustic + visual_aligned)

        # Concatenate acoustic + aligned visual
        combined = torch.cat([acoustic, visual_aligned], dim=-1)
        return self.output_proj(combined)


class GATADepDVlog(nn.Module):
    """Full GATA-Dep model for DVlog with 2 modalities."""
    def __init__(self, acoustic_dim, visual_dim,
                 d_model=256, n_heads=4, n_layers=4,
                 K=4, dropout=0.1, num_classes=2):
        super().__init__()

        self.enc_acoustic = ModalityEncoder(acoustic_dim)
        self.enc_visual   = ModalityEncoder(visual_dim)

        self.gata = GATAFusionDVlog(d_model, n_heads, K, dropout)

        self.mc_dropout = nn.Dropout(p=0.3)

        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8,
            dim_feedforward=512, dropout=dropout,
            batch_first=True)
        self.transformer = nn.TransformerEncoder(
            enc, num_layers=n_layers)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, acoustic, visual, gender, mc_samples=1):
        a = self.enc_acoustic(acoustic)   # [B, T, 256]
        v = self.enc_visual(visual)       # [B, T, 256]

        a = self.mc_dropout(a)
        v = self.mc_dropout(v)

        fused  = self.gata(a, v, gender)
        out    = self.transformer(fused).mean(dim=1)
        return self.classifier(out)


# Build model
dvlog_gata = GATADepDVlog(
    acoustic_dim = DVLOG_ACOUSTIC_DIM,
    visual_dim   = DVLOG_VISUAL_DIM,
    K=4
)
dvlog_gata = dvlog_gata.to(device)

n_params = sum(p.numel() for p in dvlog_gata.parameters()
               if p.requires_grad)
print(f"DVlog GATA-Dep parameters: {n_params:,}")

DVlog GATA-Dep parameters: 2,695,380
DVlog GATA-Dep parameters: 2,695,380


In [26]:
# ============================================================
# DVLOG SPLITS — use official fold column
# ============================================================

import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from collections import defaultdict

random.seed(42)

# ── DVLOG SPLITS — use official fold column ───────────────────
train_idx = dvlog_df[dvlog_df['fold']=='train']['index'].tolist()
val_idx   = dvlog_df[dvlog_df['fold']=='valid']['index'].tolist()
test_idx  = dvlog_df[dvlog_df['fold']=='test' ]['index'].tolist()

train_idx = dvlog_df[dvlog_df['fold']=='train']['index'].tolist()
val_idx   = dvlog_df[dvlog_df['fold']=='valid']['index'].tolist()
test_idx  = dvlog_df[dvlog_df['fold']=='test' ]['index'].tolist()

# Filter to only cached samples
train_idx = [i for i in train_idx
             if os.path.exists(
                 os.path.join(DVLOG_CACHE_DIR, f"{i}.npy"))]
val_idx   = [i for i in val_idx
             if os.path.exists(
                 os.path.join(DVLOG_CACHE_DIR, f"{i}.npy"))]
test_idx  = [i for i in test_idx
             if os.path.exists(
                 os.path.join(DVLOG_CACHE_DIR, f"{i}.npy"))]

print("Building DVlog train dataset:")
dv_train = DVlogRAMDataset(DVLOG_CACHE_DIR, train_idx)

print("\nBuilding DVlog val dataset:")
dv_val   = DVlogRAMDataset(DVLOG_CACHE_DIR, val_idx)

print("\nBuilding DVlog test dataset:")
dv_test  = DVlogRAMDataset(DVLOG_CACHE_DIR, test_idx)

# Weighted sampler for train
n_dep  = (dv_train.labels == 1).sum().item()
n_ctrl = (dv_train.labels == 0).sum().item()
total  = len(dv_train)
ratio  = n_ctrl / n_dep
print(f"\nClass ratio: {ratio:.2f}")

w_dep  = total / (2 * n_dep)
w_ctrl = total / (2 * n_ctrl)
sw = [w_dep if dv_train.labels[i].item()==1
      else w_ctrl for i in range(total)]
sampler = WeightedRandomSampler(sw, total, replacement=True)

dv_train_loader = DataLoader(
    dv_train, batch_size=32, sampler=sampler,
    num_workers=4, pin_memory=True)

dv_val_loader = DataLoader(
    dv_val, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True)

dv_test_loader = DataLoader(
    dv_test, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True)

print(f"\nTrain batches: {len(dv_train_loader)}")
print(f"Val   batches: {len(dv_val_loader)}")
print(f"Test  batches: {len(dv_test_loader)}")

Building DVlog train dataset:
  645 samples → 12741 windows dep=7744 ctrl=4997 RAM=0.25GB

Building DVlog val dataset:
  102 samples → 1851 windows dep=1182 ctrl=669 RAM=0.04GB

Building DVlog test dataset:
  212 samples → 4021 windows dep=2634 ctrl=1387 RAM=0.08GB

Class ratio: 0.65

Train batches: 399
Val   batches: 58
Test  batches: 126


In [29]:
%pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.2 MB 8.3 MB/s eta 0:00:01
   -------- ------------------------------- 1.8/8.2 MB 6.3 MB/s eta 0:00:02
   ----------- ---------------------------- 2.4/8.2 MB 5.0 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.2 MB 4.5 MB/s eta 0:00:02
   ---------------------- ----------------- 4.7/8.2 MB 4.8 MB/s eta 0:00:01
   --------------------------------- ------ 6.8/8.2 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 6.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   -- ------------------------------------- 2.4/36.6 MB 12.2 MB/s eta 0:00:03
   ---- ----------------------------------- 4.5/36.6 MB 10.7 MB/s eta 0:00:03
   -------- ------------------------------- 7.3/36.6 MB 11.9 MB/s eta 0:00:03
   ----------- -----------


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
# ============================================================
# TRAIN GATA ON DVLOG
# ============================================================

# At the top of Cell DV8, add this import
import os
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import f1_score, precision_score, recall_score
from collections import defaultdict

# Then the rest of train_gata_dvlog unchanged
# The loaders already have num_workers=0 from DV7 fix

def train_gata_dvlog(model, train_loader, val_loader,
                     n_epochs, device, save_path):

    n_dep  = (dv_train.labels == 1).sum().item()
    n_ctrl = (dv_train.labels == 0).sum().item()
    ratio  = n_ctrl / n_dep

    cw        = torch.tensor([1.0, ratio]).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)

    optimizer = torch.optim.AdamW([
        {'params': [p for n, p in model.named_parameters()
                    if 'offset_weights' in n],
         'lr': 5e-4, 'weight_decay': 0.0},
        {'params': [p for n, p in model.named_parameters()
                    if 'offset_weights' not in n],
         'lr': 1e-4, 'weight_decay': 1e-4},
    ])

    def lr_lambda(epoch):
        warmup = 10
        if epoch < warmup:
            return (epoch + 1) / warmup
        prog = (epoch - warmup) / max(n_epochs - warmup, 1)
        return max(0.1, 0.5 * (1 + np.cos(np.pi * prog)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = GradScaler()

    best_f1 = 0.0; no_improve = 0; patience = 15
    best_thresh = 0.5; start_epoch = 1

    ckpt_path = save_path.replace('.pt', '_ckpt.pt')
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device,
                          weights_only=False)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        best_f1     = ckpt['best_f1']
        no_improve  = ckpt['no_improve']
        best_thresh = ckpt.get('best_thresh', 0.5)
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {ckpt['epoch']} "
              f"F1={best_f1:.4f}")

    print(f"\n{'='*60}")
    print(f"DVlog GATA-Dep | K=4 | batch=32 | AMP | Patience={patience}")
    print(f"{'='*60}")

    for epoch in range(start_epoch, n_epochs + 1):

        # Train
        model.train()
        total_loss, n_b = 0.0, 0

        for acoustic, visual, label, gender, pid in train_loader:
            acoustic = acoustic.to(device, non_blocking=True)
            visual   = visual.to(device,   non_blocking=True)
            label    = label.to(device,    non_blocking=True)
            gender   = gender.to(device,   non_blocking=True)

            optimizer.zero_grad()
            with autocast():
                logits = model(acoustic, visual, gender)
                loss   = criterion(logits, label)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            total_loss += loss.item(); n_b += 1

        scheduler.step()
        avg_loss   = total_loss / max(n_b, 1)
        current_lr = optimizer.param_groups[0]['lr']

        # Find best threshold every 5 epochs
        if epoch % 5 == 0 or epoch <= 3:
            best_thresh = _find_thresh_dvlog(model, val_loader, device)

        # Participant-level eval
        f1, prec, rec = _eval_dvlog(
            model, val_loader, device, best_thresh)

        flag = ""
        if f1 > best_f1:
            best_f1 = f1; no_improve = 0
            torch.save(model.state_dict(), save_path)
            flag = " ← BEST"
        else:
            no_improve += 1

        torch.save({
            'epoch': epoch, 'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler':    scaler.state_dict(),
            'best_f1':   best_f1,
            'no_improve':no_improve,
            'best_thresh': best_thresh,
        }, ckpt_path)

        print(f"Ep {epoch:3d}/{n_epochs} | "
              f"Loss {avg_loss:.4f} | "
              f"F1 {f1:.4f} P {prec:.4f} R {rec:.4f} | "
              f"Thr={best_thresh:.2f} | "
              f"LR {current_lr:.1e}{flag}")

        if no_improve >= patience:
            print(f"\nEarly stop at epoch {epoch}")
            break

    print(f"\nBest DVlog F1: {best_f1:.4f}")
    return best_f1


def _find_thresh_dvlog(model, loader, device):
    """Find best threshold on DVlog val set."""
    model.eval()
    pid_probs = defaultdict(list); pid_labels = {}
    with torch.no_grad():
        for acoustic, visual, label, gender, pid in loader:
            with autocast():
                logits = model(
                    acoustic.to(device),
                    visual.to(device),
                    gender.to(device))
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            for p, pr, y in zip(
                    pid.numpy(), probs, label.numpy()):
                pid_probs[int(p)].append(float(pr))
                pid_labels[int(p)] = int(y)

    labels_all = [pid_labels[p] for p in pid_probs]
    best_f1 = 0.0; best_t = 0.5
    for t in np.arange(0.1, 0.91, 0.02):
        preds = [1 if np.mean(pid_probs[p])>=t else 0
                 for p in pid_probs]
        f = f1_score(labels_all, preds,
                     average='binary', zero_division=0)
        if f > best_f1:
            best_f1 = f; best_t = t
    return best_t


def _eval_dvlog(model, loader, device, threshold=0.5):
    """Participant-level evaluation for DVlog."""
    model.eval()
    pid_probs = defaultdict(list); pid_labels = {}
    with torch.no_grad():
        for acoustic, visual, label, gender, pid in loader:
            with autocast():
                logits = model(
                    acoustic.to(device),
                    visual.to(device),
                    gender.to(device))
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            for p, pr, y in zip(
                    pid.numpy(), probs, label.numpy()):
                pid_probs[int(p)].append(float(pr))
                pid_labels[int(p)] = int(y)

    labels_all = [pid_labels[p] for p in pid_probs]
    preds = [1 if np.mean(pid_probs[p])>=threshold else 0
             for p in pid_probs]
    f1   = f1_score(labels_all, preds,
                    average='binary', zero_division=0)
    prec = precision_score(labels_all, preds,
                           average='binary', zero_division=0)
    rec  = recall_score(labels_all, preds,
                        average='binary', zero_division=0)
    return f1, prec, rec


# Run training
dvlog_save = "./results/gata_dvlog.pt"
best_f1_dvlog = train_gata_dvlog(
    model        = dvlog_gata,
    train_loader = dv_train_loader,
    val_loader   = dv_val_loader,
    n_epochs     = 60,
    device       = device,
    save_path    = dvlog_save,
)

C:\Users\giria\AppData\Local\Temp\ipykernel_10136\67932528.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()



DVlog GATA-Dep | K=4 | batch=32 | AMP | Patience=15


RuntimeError: DataLoader worker (pid(s) 1424, 5112, 25840, 25748) exited unexpectedly

NEW  CELLS FIXED


In [31]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
from sklearn.metrics import f1_score, precision_score, recall_score
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

DVLOG_ROOT      = r"C:\Users\giria\Downloads\dvlog-dataset"
DVLOG_LABELS    = os.path.join(DVLOG_ROOT, "labels.csv")
DVLOG_CACHE_DIR = r"C:\Users\giria\Downloads\dvlog_cache"
DVLOG_RESULTS   = r"C:\Users\giria\Downloads\dvlog_results"

os.makedirs(DVLOG_CACHE_DIR, exist_ok=True)
os.makedirs(DVLOG_RESULTS,   exist_ok=True)

dvlog_df = pd.read_csv(DVLOG_LABELS)
dvlog_df['label_bin']  = (dvlog_df['label'] == 'depression').astype(int)
dvlog_df['gender_bin'] = (dvlog_df['gender'] == 'f').astype(int)

print(f"DVlog samples: {len(dvlog_df)}")
print(f"Label dist:\n{dvlog_df['label'].value_counts()}")
print(f"Fold dist:\n{dvlog_df['fold'].value_counts()}")

Device: cuda
DVlog samples: 961
Label dist:
label
depression    555
normal        406
Name: count, dtype: int64
Fold dist:
fold
train    647
test     212
valid    102
Name: count, dtype: int64


In [32]:
# Confirm file structure and get dims
sample_idx  = 0
folder      = os.path.join(DVLOG_ROOT, str(sample_idx))
a_path      = os.path.join(folder, f"{sample_idx}_acoustic.npy")
v_path      = os.path.join(folder, f"{sample_idx}_visual.npy")

acoustic = np.load(a_path).astype(np.float32)
visual   = np.load(v_path).astype(np.float32)

print(f"acoustic shape: {acoustic.shape}")
print(f"visual   shape: {visual.shape}")

DVLOG_ACOUSTIC_DIM = acoustic.shape[1]  # 25
DVLOG_VISUAL_DIM   = visual.shape[1]    # 136
DVLOG_WINDOW_SEC   = 30
DVLOG_HZ           = 1
DVLOG_WIN_FRAMES   = DVLOG_WINDOW_SEC * DVLOG_HZ  # 30

print(f"\nAcoustic dim : {DVLOG_ACOUSTIC_DIM}")
print(f"Visual dim   : {DVLOG_VISUAL_DIM}")
print(f"Window frames: {DVLOG_WIN_FRAMES}")

acoustic shape: (823, 25)
visual   shape: (823, 136)

Acoustic dim : 25
Visual dim   : 136
Window frames: 30


In [33]:
def normalize_dvlog(arr):
    mean = np.nanmean(arr, axis=0, keepdims=True)
    std  = np.nanstd (arr, axis=0, keepdims=True)
    std[std == 0] = 1.0
    arr = (arr - mean) / std
    return np.nan_to_num(
        arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def load_dvlog_sample(idx, dvlog_root):
    folder = os.path.join(dvlog_root, str(idx))
    a_path = os.path.join(folder, f"{idx}_acoustic.npy")
    v_path = os.path.join(folder, f"{idx}_visual.npy")

    if not os.path.exists(a_path) or not os.path.exists(v_path):
        return None
    try:
        acoustic = np.load(a_path).astype(np.float32)
        visual   = np.load(v_path).astype(np.float32)
        if acoustic.ndim == 1: acoustic = acoustic.reshape(-1, 1)
        if visual.ndim == 1:   visual   = visual.reshape(-1, 1)
        return {'acoustic': acoustic, 'visual': visual}
    except Exception as e:
        print(f"  ERROR {idx}: {e}")
        return None


def make_dvlog_windows(features, win_frames=DVLOG_WIN_FRAMES):
    acoustic = normalize_dvlog(features['acoustic'])
    visual   = normalize_dvlog(features['visual'])
    min_len  = min(len(acoustic), len(visual))
    acoustic = acoustic[:min_len]
    visual   = visual[:min_len]
    n_wins   = min_len // win_frames
    windows  = []
    for i in range(n_wins):
        s = i * win_frames
        windows.append({
            'acoustic': acoustic[s: s + win_frames],
            'visual':   visual  [s: s + win_frames],
        })
    return windows

print("Load + window functions defined ✓")

Load + window functions defined ✓


In [34]:
def cache_dvlog(dvlog_df, dvlog_root, cache_dir):
    already = set(os.listdir(cache_dir))
    total_windows = 0
    failed = []

    for _, row in dvlog_df.iterrows():
        idx   = int(row['index'])
        fname = f"{idx}.npy"
        if fname in already:
            continue

        feats = load_dvlog_sample(idx, dvlog_root)
        if feats is None:
            failed.append(idx); continue

        wins = make_dvlog_windows(feats)
        if not wins:
            failed.append(idx); continue

        np.save(
            os.path.join(cache_dir, fname),
            {
                'acoustic': np.stack([w['acoustic'] for w in wins]),
                'visual':   np.stack([w['visual']   for w in wins]),
                'label':    np.array([int(row['label_bin'])]  * len(wins)),
                'gender':   np.array([int(row['gender_bin'])] * len(wins)),
            }
        )
        total_windows += len(wins)

    size_mb = sum(
        os.path.getsize(os.path.join(cache_dir, f))
        for f in os.listdir(cache_dir)
    ) / 1e6
    print(f"Cached {total_windows} windows | "
          f"Failed: {len(failed)} | Size: {size_mb:.1f} MB")

print("Caching DVlog...")
cache_dvlog(dvlog_df, DVLOG_ROOT, DVLOG_CACHE_DIR)

Caching DVlog...
Cached 18613 windows | Failed: 2 | Size: 360.4 MB


In [35]:
class DVlogRAMDataset(Dataset):
    def __init__(self, cache_dir, indices):
        acoustic_list = []; visual_list = []
        labels = []; genders = []; pids_all = []

        for idx in indices:
            path = os.path.join(cache_dir, f"{idx}.npy")
            if not os.path.exists(path):
                continue
            try:
                data = np.load(path, allow_pickle=True).item()
                n = len(data['label'])
                acoustic_list.append(data['acoustic'])
                visual_list.append(data['visual'])
                labels.extend(data['label'].tolist())
                genders.extend(data['gender'].tolist())
                pids_all.extend([idx] * n)
            except Exception as e:
                print(f"  SKIP {idx}: {e}")

        self.acoustic = torch.FloatTensor(np.concatenate(acoustic_list))
        self.visual   = torch.FloatTensor(np.concatenate(visual_list))
        self.labels   = torch.tensor(labels,   dtype=torch.long)
        self.genders  = torch.tensor(genders,  dtype=torch.long)
        self.pids     = torch.tensor(pids_all, dtype=torch.long)

        dep  = self.labels.sum().item()
        ctrl = len(self.labels) - dep
        ram  = (self.acoustic.nbytes + self.visual.nbytes) / 1e9
        print(f"  {len(indices)} samples → {len(self.labels)} windows "
              f"dep={int(dep)} ctrl={ctrl} RAM={ram:.3f}GB")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (self.acoustic[idx], self.visual[idx],
                self.labels[idx], self.genders[idx],
                self.pids[idx])

print("DVlogRAMDataset defined ✓")

DVlogRAMDataset defined ✓


In [36]:
class ModalityEncoderDV(nn.Module):
    """Same as ModalityEncoder but standalone for DVlog."""
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )
    def forward(self, x):
        B, T, D = x.shape
        return self.net(x.reshape(B*T, D)).reshape(B, T, 256)


class GATAFusionDVlog(nn.Module):
    def __init__(self, d_model=256, n_heads=4, K=4, dropout=0.1):
        super().__init__()
        self.K         = K
        self.n_offsets = 2 * K + 1

        self.attn_visual = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)

        self.offset_weights = nn.Parameter(
            torch.zeros(2, 1, self.n_offsets))

        self.norm_visual = nn.LayerNorm(d_model)
        self.output_proj = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(), nn.Dropout(dropout))

    def _shift(self, x, k):
        if k == 0: return x
        B, T, D = x.shape
        out = torch.zeros_like(x)
        if k > 0:
            out[:, k:, :]    = x[:, :T-k, :]
        else:
            ak = abs(k)
            out[:, :T-ak, :] = x[:, ak:, :]
        return out

    def forward(self, acoustic, visual, gender):
        B, T, D = acoustic.shape
        n = self.n_offsets

        shifted = torch.stack(
            [self._shift(visual, k)
             for k in range(-self.K, self.K+1)],
            dim=1).reshape(B * n, T, D)

        q_rep = (acoustic.unsqueeze(1)
                         .expand(-1, n, -1, -1)
                         .reshape(B * n, T, D))

        attn_out, _ = self.attn_visual(
            query=q_rep, key=shifted, value=shifted)
        attn_out = attn_out.view(B, n, T, D)

        w_all    = torch.softmax(
            self.offset_weights[:, 0, :], dim=-1)
        w_sample = w_all[gender].unsqueeze(-1).unsqueeze(-1)

        visual_aligned = (attn_out * w_sample).sum(dim=1)
        visual_aligned = self.norm_visual(acoustic + visual_aligned)

        combined = torch.cat([acoustic, visual_aligned], dim=-1)
        return self.output_proj(combined)


class GATADepDVlog(nn.Module):
    def __init__(self, acoustic_dim, visual_dim,
                 d_model=256, n_heads=4, n_layers=4,
                 K=4, dropout=0.1, num_classes=2):
        super().__init__()
        self.enc_acoustic = ModalityEncoderDV(acoustic_dim)
        self.enc_visual   = ModalityEncoderDV(visual_dim)
        self.gata         = GATAFusionDVlog(d_model, n_heads, K, dropout)
        self.mc_dropout   = nn.Dropout(p=0.3)

        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8,
            dim_feedforward=512, dropout=dropout,
            batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, acoustic, visual, gender, mc_samples=1):
        a = self.enc_acoustic(acoustic)
        v = self.enc_visual(visual)
        a = self.mc_dropout(a)
        v = self.mc_dropout(v)
        fused  = self.gata(a, v, gender)
        out    = self.transformer(fused).mean(dim=1)
        return self.classifier(out)


# Build
dvlog_gata = GATADepDVlog(
    acoustic_dim=DVLOG_ACOUSTIC_DIM,
    visual_dim=DVLOG_VISUAL_DIM,
    K=4)
dvlog_gata = dvlog_gata.to(device)

n_params = sum(p.numel() for p in dvlog_gata.parameters()
               if p.requires_grad)
print(f"DVlog GATA-Dep parameters: {n_params:,}")

DVlog GATA-Dep parameters: 2,562,070


In [37]:
train_idx = dvlog_df[dvlog_df['fold']=='train']['index'].tolist()
val_idx   = dvlog_df[dvlog_df['fold']=='valid']['index'].tolist()
test_idx  = dvlog_df[dvlog_df['fold']=='test' ]['index'].tolist()

train_idx = [i for i in train_idx
             if os.path.exists(os.path.join(DVLOG_CACHE_DIR, f"{i}.npy"))]
val_idx   = [i for i in val_idx
             if os.path.exists(os.path.join(DVLOG_CACHE_DIR, f"{i}.npy"))]
test_idx  = [i for i in test_idx
             if os.path.exists(os.path.join(DVLOG_CACHE_DIR, f"{i}.npy"))]

print("Building DVlog train dataset:")
dv_train = DVlogRAMDataset(DVLOG_CACHE_DIR, train_idx)
print("\nBuilding DVlog val dataset:")
dv_val   = DVlogRAMDataset(DVLOG_CACHE_DIR, val_idx)
print("\nBuilding DVlog test dataset:")
dv_test  = DVlogRAMDataset(DVLOG_CACHE_DIR, test_idx)

n_dep  = (dv_train.labels == 1).sum().item()
n_ctrl = (dv_train.labels == 0).sum().item()
total  = len(dv_train)
ratio  = n_ctrl / n_dep

w_dep  = total / (2 * n_dep)
w_ctrl = total / (2 * n_ctrl)
sw = [w_dep if dv_train.labels[i].item()==1
      else w_ctrl for i in range(total)]
sampler = WeightedRandomSampler(sw, total, replacement=True)

# num_workers=0 and pin_memory=False — required on Windows
dv_train_loader = DataLoader(
    dv_train, batch_size=32, sampler=sampler,
    num_workers=0, pin_memory=False)

dv_val_loader = DataLoader(
    dv_val, batch_size=32, shuffle=False,
    num_workers=0, pin_memory=False)

dv_test_loader = DataLoader(
    dv_test, batch_size=32, shuffle=False,
    num_workers=0, pin_memory=False)

print(f"\nTrain batches: {len(dv_train_loader)}")
print(f"Val   batches: {len(dv_val_loader)}")
print(f"Test  batches: {len(dv_test_loader)}")

Building DVlog train dataset:
  645 samples → 12741 windows dep=7744 ctrl=4997 RAM=0.246GB

Building DVlog val dataset:
  102 samples → 1851 windows dep=1182 ctrl=669 RAM=0.036GB

Building DVlog test dataset:
  212 samples → 4021 windows dep=2634 ctrl=1387 RAM=0.078GB

Train batches: 399
Val   batches: 58
Test  batches: 126


In [38]:
def _find_thresh_dvlog(model, loader, device):
    model.eval()
    pid_probs = defaultdict(list); pid_labels = {}
    with torch.no_grad():
        for acoustic, visual, label, gender, pid in loader:
            with autocast('cuda'):
                logits = model(
                    acoustic.to(device),
                    visual.to(device),
                    gender.to(device))
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            for p, pr, y in zip(pid.numpy(), probs, label.numpy()):
                pid_probs[int(p)].append(float(pr))
                pid_labels[int(p)] = int(y)

    labels_all = [pid_labels[p] for p in pid_probs]
    best_f1 = 0.0; best_t = 0.5
    for t in np.arange(0.1, 0.91, 0.02):
        preds = [1 if np.mean(pid_probs[p])>=t else 0
                 for p in pid_probs]
        f = f1_score(labels_all, preds,
                     average='binary', zero_division=0)
        if f > best_f1:
            best_f1 = f; best_t = t
    return best_t


def _eval_dvlog(model, loader, device, threshold=0.5):
    model.eval()
    pid_probs = defaultdict(list); pid_labels = {}
    with torch.no_grad():
        for acoustic, visual, label, gender, pid in loader:
            with autocast('cuda'):
                logits = model(
                    acoustic.to(device),
                    visual.to(device),
                    gender.to(device))
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            for p, pr, y in zip(pid.numpy(), probs, label.numpy()):
                pid_probs[int(p)].append(float(pr))
                pid_labels[int(p)] = int(y)

    labels_all = [pid_labels[p] for p in pid_probs]
    preds = [1 if np.mean(pid_probs[p])>=threshold else 0
             for p in pid_probs]
    f1   = f1_score(labels_all, preds,
                    average='binary', zero_division=0)
    prec = precision_score(labels_all, preds,
                           average='binary', zero_division=0)
    rec  = recall_score(labels_all, preds,
                        average='binary', zero_division=0)
    return f1, prec, rec


def train_gata_dvlog(model, train_loader, val_loader,
                     n_epochs, device, save_path):

    n_dep  = (dv_train.labels == 1).sum().item()
    n_ctrl = (dv_train.labels == 0).sum().item()
    ratio  = n_ctrl / n_dep

    cw        = torch.tensor([1.0, ratio]).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)

    optimizer = torch.optim.AdamW([
        {'params': [p for n, p in model.named_parameters()
                    if 'offset_weights' in n],
         'lr': 5e-4, 'weight_decay': 0.0},
        {'params': [p for n, p in model.named_parameters()
                    if 'offset_weights' not in n],
         'lr': 1e-4, 'weight_decay': 1e-4},
    ])

    def lr_lambda(epoch):
        warmup = 10
        if epoch < warmup:
            return (epoch + 1) / warmup
        prog = (epoch - warmup) / max(n_epochs - warmup, 1)
        return max(0.1, 0.5 * (1 + np.cos(np.pi * prog)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = GradScaler('cuda')   # ← new syntax for PyTorch 2.6

    best_f1 = 0.0; no_improve = 0; patience = 15
    best_thresh = 0.5; start_epoch = 1

    ckpt_path = save_path.replace('.pt', '_ckpt.pt')
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device,
                          weights_only=False)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        best_f1     = ckpt['best_f1']
        no_improve  = ckpt['no_improve']
        best_thresh = ckpt.get('best_thresh', 0.5)
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {ckpt['epoch']} F1={best_f1:.4f}")

    print(f"\n{'='*60}")
    print(f"DVlog GATA-Dep | K=4 | batch=32 | AMP | Patience={patience}")
    print(f"{'='*60}")

    for epoch in range(start_epoch, n_epochs + 1):

        model.train()
        total_loss, n_b = 0.0, 0

        for acoustic, visual, label, gender, pid in train_loader:
            acoustic = acoustic.to(device)
            visual   = visual.to(device)
            label    = label.to(device)
            gender   = gender.to(device)

            optimizer.zero_grad()
            with autocast('cuda'):
                logits = model(acoustic, visual, gender)
                loss   = criterion(logits, label)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            total_loss += loss.item(); n_b += 1

        scheduler.step()
        avg_loss   = total_loss / max(n_b, 1)
        current_lr = optimizer.param_groups[0]['lr']

        if epoch % 5 == 0 or epoch <= 3:
            best_thresh = _find_thresh_dvlog(
                model, val_loader, device)

        f1, prec, rec = _eval_dvlog(
            model, val_loader, device, best_thresh)

        flag = ""
        if f1 > best_f1:
            best_f1 = f1; no_improve = 0
            torch.save(model.state_dict(), save_path)
            flag = " ← BEST"
        else:
            no_improve += 1

        torch.save({
            'epoch':       epoch,
            'model':       model.state_dict(),
            'optimizer':   optimizer.state_dict(),
            'scheduler':   scheduler.state_dict(),
            'scaler':      scaler.state_dict(),
            'best_f1':     best_f1,
            'no_improve':  no_improve,
            'best_thresh': best_thresh,
        }, ckpt_path)

        print(f"Ep {epoch:3d}/{n_epochs} | "
              f"Loss {avg_loss:.4f} | "
              f"F1 {f1:.4f} P {prec:.4f} R {rec:.4f} | "
              f"Thr={best_thresh:.2f} | "
              f"LR {current_lr:.1e}{flag}")

        if no_improve >= patience:
            print(f"\nEarly stop at epoch {epoch}")
            break

    print(f"\nBest DVlog val F1: {best_f1:.4f}")
    return best_f1


dvlog_save = os.path.join(DVLOG_RESULTS, "gata_dvlog.pt")
best_f1_dvlog = train_gata_dvlog(
    model        = dvlog_gata,
    train_loader = dv_train_loader,
    val_loader   = dv_val_loader,
    n_epochs     = 60,
    device       = device,
    save_path    = dvlog_save,
)


DVlog GATA-Dep | K=4 | batch=32 | AMP | Patience=15
Ep   1/60 | Loss 0.6559 | F1 0.7612 P 0.6623 R 0.8947 | Thr=0.30 | LR 1.0e-04 ← BEST
Ep   2/60 | Loss 0.6333 | F1 0.7397 P 0.6067 R 0.9474 | Thr=0.24 | LR 1.5e-04
Ep   3/60 | Loss 0.6205 | F1 0.7517 P 0.6087 R 0.9825 | Thr=0.18 | LR 2.0e-04
Ep   4/60 | Loss 0.5993 | F1 0.7215 P 0.5644 R 1.0000 | Thr=0.18 | LR 2.5e-04
Ep   5/60 | Loss 0.5830 | F1 0.7559 P 0.6857 R 0.8421 | Thr=0.22 | LR 3.0e-04
Ep   6/60 | Loss 0.5655 | F1 0.7576 P 0.6667 R 0.8772 | Thr=0.22 | LR 3.5e-04
Ep   7/60 | Loss 0.5450 | F1 0.6981 P 0.7551 R 0.6491 | Thr=0.22 | LR 4.0e-04
Ep   8/60 | Loss 0.5390 | F1 0.7222 P 0.5977 R 0.9123 | Thr=0.22 | LR 4.5e-04
Ep   9/60 | Loss 0.5194 | F1 0.7097 P 0.5612 R 0.9649 | Thr=0.22 | LR 5.0e-04
Ep  10/60 | Loss 0.4980 | F1 0.7869 P 0.7385 R 0.8421 | Thr=0.38 | LR 5.0e-04 ← BEST
Ep  11/60 | Loss 0.4854 | F1 0.7257 P 0.7321 R 0.7193 | Thr=0.38 | LR 5.0e-04
Ep  12/60 | Loss 0.4584 | F1 0.6931 P 0.7955 R 0.6140 | Thr=0.38 | LR 5.0e-

In [39]:
dvlog_gata.load_state_dict(
    torch.load(dvlog_save, map_location=device,
               weights_only=False))
dvlog_gata = dvlog_gata.to(device)

best_thresh_dv = _find_thresh_dvlog(dvlog_gata, dv_val_loader, device)

f1_test,  prec_test,  rec_test  = _eval_dvlog(
    dvlog_gata, dv_test_loader, device, best_thresh_dv)
f1_val,   prec_val,   rec_val   = _eval_dvlog(
    dvlog_gata, dv_val_loader,  device, best_thresh_dv)

print("\n" + "="*55)
print("DVLOG FINAL RESULTS (Participant-Level)")
print("="*55)
print(f"{'Metric':<12} {'Val':>8} {'Test':>8}")
print("-"*30)
print(f"{'F1':<12} {f1_val:>8.4f} {f1_test:>8.4f}")
print(f"{'Precision':<12} {prec_val:>8.4f} {prec_test:>8.4f}")
print(f"{'Recall':<12} {rec_val:>8.4f} {rec_test:>8.4f}")
print(f"\nThreshold: {best_thresh_dv:.2f}")

print("\n" + "="*55)
print("CROSS-DATASET TABLE FOR PAPER")
print("="*55)
print(f"{'Dataset':<12} {'F1 Val':>8} {'F1 Test':>8}")
print("-"*30)
print(f"{'DAIC-WOZ':<12} {'0.6286':>8} {'—':>8}")
print(f"{'DVlog':<12} {f1_val:>8.4f} {f1_test:>8.4f}")


DVLOG FINAL RESULTS (Participant-Level)
Metric            Val     Test
------------------------------
F1             0.8092   0.7986
Precision      0.7162   0.6882
Recall         0.9298   0.9512

Threshold: 0.24

CROSS-DATASET TABLE FOR PAPER
Dataset        F1 Val  F1 Test
------------------------------
DAIC-WOZ       0.6286        —
DVlog          0.8092   0.7986
